In [19]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python executable:
C:\Users\DELL\anaconda3\envs\qafza-mlops\python.exe

Python version:
3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:37:13) [MSC v.1942 64 bit (AMD64)]


# Experiment 2 – Improved Feature Engineering

In [2]:
#Inspect Available Columns
import pandas as pd

train = pd.read_csv("../artifacts/train.csv")

print("Shape:", train.shape)
print("\nColumns:")
print(train.columns.tolist())

Shape: (67533, 28)

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'item_count', 'total_price', 'total_freight', 'payment_count', 'total_payment', 'payment_installments', 'review_count', 'average_review_score', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'unique_products', 'average_product_weight', 'average_product_photos', 'delivery_days', 'delivery_label']


## 1. Load the Data

I load the training, validation, and test datasets created in Notebook 3.

In [5]:
import pandas as pd

train = pd.read_csv("../artifacts/train.csv")
validation = pd.read_csv("../artifacts/validation.csv")
test = pd.read_csv("../artifacts/test.csv")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67533, 28)
Validation: (14471, 28)
Test: (14472, 28)


## 2. Create Improved Prediction-Time Features

In [6]:
# Convert relevant date columns to datetime

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_estimated_delivery_date"
]

for df in [train, validation, test]:
    for col in date_columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")


# Create estimated delivery duration
for df in [train, validation, test]:
    df["estimated_delivery_days"] = (
        df["order_estimated_delivery_date"]
        - df["order_purchase_timestamp"]
    ).dt.total_seconds() / (24 * 60 * 60)


print("Estimated delivery feature created successfully.")

print("\nTrain sample:")
print(
    train[
        [
            "order_purchase_timestamp",
            "order_estimated_delivery_date",
            "estimated_delivery_days"
        ]
    ].head()
)

Estimated delivery feature created successfully.

Train sample:
  order_purchase_timestamp order_estimated_delivery_date  \
0      2016-09-15 12:16:38                    2016-10-04   
1      2016-10-03 09:44:50                    2016-10-27   
2      2016-10-03 16:56:50                    2016-11-07   
3      2016-10-03 21:01:41                    2016-11-25   
4      2016-10-03 21:13:36                    2016-11-29   

   estimated_delivery_days  
0                18.488449  
1                23.593866  
2                34.293866  
3                52.123831  
4                56.115556  


In [7]:
# Create approval delay feature

for df in [train, validation, test]:
    df["approval_delay_hours"] = (
        df["order_approved_at"]
        - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 3600


print("Approval delay feature created successfully.")

print("\nTrain sample:")
print(
    train[
        [
            "order_purchase_timestamp",
            "order_approved_at",
            "approval_delay_hours"
        ]
    ].head()
)

Approval delay feature created successfully.

Train sample:
  order_purchase_timestamp   order_approved_at  approval_delay_hours
0      2016-09-15 12:16:38 2016-09-15 12:16:38              0.000000
1      2016-10-03 09:44:50 2016-10-06 15:50:54             78.101111
2      2016-10-03 16:56:50 2016-10-06 16:03:44             71.115000
3      2016-10-03 21:01:41 2016-10-04 10:18:57             13.287778
4      2016-10-03 21:13:36 2016-10-05 03:11:49             29.970278


In [8]:
# Create calendar-based features

for df in [train, validation, test]:
    df["order_day_of_week"] = df["order_purchase_timestamp"].dt.dayofweek
    df["order_month"] = df["order_purchase_timestamp"].dt.month
    df["is_weekend"] = (
        df["order_purchase_timestamp"].dt.dayofweek >= 5
    ).astype(int)


print("Calendar features created successfully.")

print("\nTrain sample:")
print(
    train[
        [
            "order_purchase_timestamp",
            "order_day_of_week",
            "order_month",
            "is_weekend"
        ]
    ].head()
)

Calendar features created successfully.

Train sample:
  order_purchase_timestamp  order_day_of_week  order_month  is_weekend
0      2016-09-15 12:16:38                  3            9           0
1      2016-10-03 09:44:50                  0           10           0
2      2016-10-03 16:56:50                  0           10           0
3      2016-10-03 21:01:41                  0           10           0
4      2016-10-03 21:13:36                  0           10           0


In [10]:
# Create geographical distance feature

import numpy as np
import pandas as pd

# Load geolocation data
geolocation = pd.read_csv(
    "../olist_geolocation_dataset.csv"
)

# Standardize geolocation ZIP-code prefixes
geolocation["geolocation_zip_code_prefix"] = (
    pd.to_numeric(
        geolocation["geolocation_zip_code_prefix"],
        errors="coerce"
    )
    .astype("Int64")
    .astype(str)
    .replace("<NA>", np.nan)
    .str.zfill(5)
)

# Average coordinates for each ZIP-code prefix
geo_by_zip = (
    geolocation
    .groupby("geolocation_zip_code_prefix", as_index=False)
    [["geolocation_lat", "geolocation_lng"]]
    .mean()
)

print("Geolocation records:", geolocation.shape)
print("Unique ZIP prefixes:", geo_by_zip.shape[0])


# Function to standardize ZIP-code prefixes
def standardize_zip(series):
    return (
        pd.to_numeric(series, errors="coerce")
        .astype("Int64")
        .astype(str)
        .replace("<NA>", np.nan)
        .str.zfill(5)
    )


# Function to add distance to one dataset
def add_distance_feature(df):

    df = df.copy()

    # Standardize customer and seller ZIP prefixes
    df["customer_zip_temp"] = standardize_zip(
        df["customer_zip_code_prefix"]
    )

    df["seller_zip_temp"] = standardize_zip(
        df["seller_zip_code_prefix"]
    )

    # Add customer coordinates
    customer_geo = geo_by_zip.rename(
        columns={
            "geolocation_zip_code_prefix": "customer_zip_temp",
            "geolocation_lat": "customer_lat",
            "geolocation_lng": "customer_lng"
        }
    )

    df = df.merge(
        customer_geo,
        on="customer_zip_temp",
        how="left"
    )

    # Add seller coordinates
    seller_geo = geo_by_zip.rename(
        columns={
            "geolocation_zip_code_prefix": "seller_zip_temp",
            "geolocation_lat": "seller_lat",
            "geolocation_lng": "seller_lng"
        }
    )

    df = df.merge(
        seller_geo,
        on="seller_zip_temp",
        how="left"
    )

    # Convert coordinates to radians
    lat1 = np.radians(df["customer_lat"])
    lon1 = np.radians(df["customer_lng"])

    lat2 = np.radians(df["seller_lat"])
    lon2 = np.radians(df["seller_lng"])

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    df["distance_km"] = (
        6371
        * 2
        * np.arcsin(np.sqrt(a))
    )

    # Remove temporary columns
    df.drop(
        columns=[
            "customer_zip_temp",
            "seller_zip_temp",
            "customer_lat",
            "customer_lng",
            "seller_lat",
            "seller_lng"
        ],
        inplace=True
    )

    return df


# Apply the feature correctly to all three datasets
train = add_distance_feature(train)
validation = add_distance_feature(validation)
test = add_distance_feature(test)


print("\nDistance feature created successfully.")

print("\nTrain distance statistics:")
print(train["distance_km"].describe())

print("\nMissing distance values:")
print("Train:", train["distance_km"].isna().sum())
print("Validation:", validation["distance_km"].isna().sum())
print("Test:", test["distance_km"].isna().sum())

Geolocation records: (1000163, 5)
Unique ZIP prefixes: 19015

Distance feature created successfully.

Train distance statistics:
count    67188.000000
mean       614.685730
std        594.990683
min          0.000000
25%        219.420601
50%        448.583670
75%        811.254211
max       5338.619521
Name: distance_km, dtype: float64

Missing distance values:
Train: 345
Validation: 74
Test: 59


## 3. Select Experiment 2 Features

In [12]:
# Create geographical same-state feature

for df in [train, validation, test]:
    df["same_state"] = (
        df["customer_state"] == df["seller_state"]
    ).astype(int)

print("same_state feature created successfully.")

print("\nSame-state distribution in training data:")
print(train["same_state"].value_counts())

same_state feature created successfully.

Same-state distribution in training data:
same_state
0    44448
1    23085
Name: count, dtype: int64


In [13]:
# Select leakage-safe features for Experiment 2

feature_cols = [
    # Order and payment information
    "item_count",
    "total_price",
    "total_freight",
    "payment_count",
    "total_payment",
    "payment_installments",

    # Product information
    "unique_products",
    "average_product_weight",
    "average_product_photos",

    # Customer / seller location
    "customer_state",
    "seller_state",
    "customer_zip_code_prefix",
    "seller_zip_code_prefix",

    # Engineered geographical feature
    "same_state",
    "distance_km",

    # Engineered temporal features
    "estimated_delivery_days",
    "approval_delay_hours",
    "order_day_of_week",
    "order_month",
    "is_weekend"
]

target = "delivery_label"

X_train = train[feature_cols].copy()
y_train = train[target].copy()

X_validation = validation[feature_cols].copy()
y_validation = validation[target].copy()

X_test = test[feature_cols].copy()
y_test = test[target].copy()

print("Number of selected features:", len(feature_cols))
print("Train features:", X_train.shape)
print("Validation features:", X_validation.shape)
print("Test features:", X_test.shape)

Number of selected features: 20
Train features: (67533, 20)
Validation features: (14471, 20)
Test features: (14472, 20)


## 4. Preprocessing

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Categorical features
categorical_features = [
    "customer_state",
    "seller_state",
    "order_day_of_week",
    "order_month"
]

# Numerical features
numerical_features = [
    col for col in feature_cols
    if col not in categorical_features
]

# Numerical preprocessing
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the fitted preprocessor to validation and test
X_validation_processed = preprocessor.transform(X_validation)
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing completed successfully.")

print("\nProcessed feature shapes:")
print("Train:", X_train_processed.shape)
print("Validation:", X_validation_processed.shape)
print("Test:", X_test_processed.shape)

Preprocessing completed successfully.

Processed feature shapes:
Train: (67533, 84)
Validation: (14471, 84)
Test: (14472, 84)


## 5. Save Experiment 2 Preprocessor

In [15]:
import joblib

joblib.dump(
    preprocessor,
    "../artifacts/preprocessor_experiment2.joblib"
)

print("Experiment 2 preprocessor saved successfully.")

Experiment 2 preprocessor saved successfully.


## 6. Save Experiment 2 Feature Data

In [16]:
# Save Experiment 2 processed feature matrices and target labels

import numpy as np
from scipy.sparse import save_npz

save_npz(
    "../artifacts/X_train_experiment2.npz",
    X_train_processed
)

save_npz(
    "../artifacts/X_validation_experiment2.npz",
    X_validation_processed
)

save_npz(
    "../artifacts/X_test_experiment2.npz",
    X_test_processed
)

np.save(
    "../artifacts/y_train_experiment2.npy",
    y_train.to_numpy()
)

np.save(
    "../artifacts/y_validation_experiment2.npy",
    y_validation.to_numpy()
)

np.save(
    "../artifacts/y_test_experiment2.npy",
    y_test.to_numpy()
)

print("Experiment 2 feature data saved successfully.")

Experiment 2 feature data saved successfully.


## 7. Save Experiment 2 Feature List

In [17]:
# Save the Experiment 2 feature list

with open(
    "../artifacts/feature_list_experiment2.txt",
    "w",
    encoding="utf-8"
) as f:
    for feature in feature_cols:
        f.write(feature + "\n")

print("Experiment 2 feature list saved successfully.")
print("Number of original features:", len(feature_cols))

Experiment 2 feature list saved successfully.
Number of original features: 20


## 8. Verify Experiment 2 Artifacts

In [18]:
import os

artifacts = [
    "preprocessor_experiment2.joblib",
    "X_train_experiment2.npz",
    "X_validation_experiment2.npz",
    "X_test_experiment2.npz",
    "y_train_experiment2.npy",
    "y_validation_experiment2.npy",
    "y_test_experiment2.npy",
    "feature_list_experiment2.txt"
]

print("Experiment 2 Artifacts:")

for artifact in artifacts:
    path = os.path.join("../artifacts", artifact)
    print(f"{artifact}: {os.path.exists(path)}")

Experiment 2 Artifacts:
preprocessor_experiment2.joblib: True
X_train_experiment2.npz: True
X_validation_experiment2.npz: True
X_test_experiment2.npz: True
y_train_experiment2.npy: True
y_validation_experiment2.npy: True
y_test_experiment2.npy: True
feature_list_experiment2.txt: True
